In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
import numpy as np
from torchvision.transforms.v2 import RandAugment
import torch
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from multiheadmodel import MultiHeadModel
from utils import deterministic, train

In [4]:
backbone= BackBone()
backbone.load_state_dict(torch.load("../models/weights/backbone.pth"))
model = MultiHeadModel(backbone)
for param in model.backbone.parameters(): 
    param.requires_grad = False

/Users/marcelokaucher/miniforge3/envs/vision/lib/python3.14/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/marcelokaucher/miniforge3/envs/vision/lib/python3.14/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Comentario de claude para acelerar el entrenamiento de las cabezas:

Si el backbone está congelado, podés pre-calcular los embeddings una sola vez y entrenar solo sobre ellos — mucho más rápido

In [ ]:
dataloaders = get_data_loaders(batch_size=512)
# ver device

/Users/marcelokaucher/miniforge3/envs/vision/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


#### Fine Tuning Naive

In [7]:
for i in range(5):
    model.add_head(i, 2)
    train_data = dataloaders[i][0]
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    epochs = 2

    train(model, train_data, optimizer, criterion, f"task{i}_training", epochs, task_number=i)
    
    eval_data = dataloaders[i][1]
    acc = evaluate(model, eval_data, task_number=i)
    print(f"Task {i} Accuracy: {acc:.4f}")

Epochs:   0%|          | 0/2 [00:00<?, ?epoch/s]

Epoch 1/2:   0%|          | 0/18 [00:00<?, ?batch/s]

5.137182712554932
4.934484481811523
5.13041353225708
4.943486213684082
4.959320068359375
4.573102951049805
5.066054821014404
4.531225681304932
4.55618143081665
3.969977378845215
4.5038933753967285
4.082686424255371
3.927117109298706
3.800126075744629
3.837981700897217
3.822103261947632
3.9175446033477783
3.1395111083984375


Epoch 2/2:   0%|          | 0/18 [00:00<?, ?batch/s]

3.1958227157592773
2.705505847930908
2.8064534664154053
3.1506710052490234
2.709683656692505
2.2970354557037354
2.228915214538574
2.078878879547119
2.2095842361450195
1.789831519126892
1.8140568733215332
1.6211235523223877
1.5140174627304077
1.3852614164352417
1.2792458534240723
1.21891188621521
1.2080150842666626
1.1881458759307861


NameError: name 'evaluate' is not defined